# Imports

In [1]:
import base64  # 바이너리 파일(예: 이미지)을 ASCII 문자열로 인코딩
from src.utils import openai_client

# 웹 상의 이미지에 대한 설명 생성

In [6]:
# https://picsum.photos/400/300 여기에서 랜덤하게 보여주는 이미지에 대한 설명
# 이미지 설명을 요청하는 메시지 프롬프트를 작성
messages = [
    {
        'role': 'user',
        'content': [
            {'type': 'text', 'text': '이 이미지를 설명해줘.'},
            {
                'type': 'image_url',
                'image_url': {'url': 'https://fastly.picsum.photos/id/551/400/300.jpg?hmac=NORg5QiPAjm-Z5V8MvKs05Ihdq4nTTpMOkqs3AeyWzU'}
            }
        ]
    }
]

In [7]:
response = openai_client.chat.completions.create(
    model='gpt-5.4-mini',
    messages=messages
)

In [8]:
response

ChatCompletion(id='chatcmpl-DPJilOChYOBkkNBg0TCqN7eNajkT8', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='황금빛 노을이 드리운 넓은 들판을 흰색 밴 한 대가 흙길을 따라 달리고 있는 모습입니다. 바퀴 뒤로 먼지가 일어나고 있고, 멀리 산맥이 실루엣처럼 보입니다. 하늘에는 구름이 길게 드리워져 있으며, 전체적으로 고요하면서도 여행의 느낌이 나는 풍경입니다.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1774926379, model='gpt-5.4-mini-2026-03-17', object='chat.completion', service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=92, prompt_tokens=171, total_tokens=263, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

In [9]:
print(response.choices[0].message.content)

황금빛 노을이 드리운 넓은 들판을 흰색 밴 한 대가 흙길을 따라 달리고 있는 모습입니다. 바퀴 뒤로 먼지가 일어나고 있고, 멀리 산맥이 실루엣처럼 보입니다. 하늘에는 구름이 길게 드리워져 있으며, 전체적으로 고요하면서도 여행의 느낌이 나는 풍경입니다.


# 로컬 이미지 파일의 설명 요청

* 이미지와 같은 바이너리 데이터는 텍스트 형식으로 변환해서 GPT 요청을 보내야 함.
* base64 라이브러리: 이진(binary) 데이터를 ASCII 문자로 인코딩(변환).
    * 전송/저장 호환성 유지
    * 데이터 손상 방지
    * 프로토콜 호환성

In [10]:
def base64encode_image(image_file):
    # 이미지 파일을 이진데이터 읽기 모드로 오픈
    with open(file=image_file, mode='rb') as f:
        data = f.read()  # 파일 내용 전체 읽기

        # 이진 데이터를 base64 인코딩을 해서 ASCII 문자열로 변환, utf-8 인코딩.
        return base64.b64encode(data).decode(encoding='utf-8')

In [11]:
# 이미지 파일(jpg)들이 저장된 경로
image_1 = './images/image_1.jpg'
image_2 = './images/image_2.jpg'

In [12]:
b64_image_1 = base64encode_image(image_1)

In [14]:
b64_image_1[:100]  # 이미지 이진 데이터가 ASCII 문자열로 인코딩됨.

'/9j/4QDgRXhpZgAASUkqAAgAAAAGABIBAwABAAAAAQAAABoBBQABAAAAVgAAABsBBQABAAAAXgAAACgBAwABAAAAAgAAABMCAwAB'

In [15]:
b64_image_1[-100:]

'FrcCdShEvOIAAoOP1yd7BPnBqCWY5WSpECCI095rwwuHgwBDkNfvjmG+3bgIJ1cGm7Dh8jVdYZWigPWVh8Bho2u28WDecrY+M//Z'

In [16]:
b64_image_2 = base64encode_image(image_2)

In [21]:
# 로컬 이미지 데이터를 GPT 서버로 전송
messages = [
    {
        'role': 'user',
        'content': [
            {'type': 'text', 'text': '이 이미지에 대해서 설명해줘'},
            {
                'type': 'image_url',
                'image_url': {'url': f'data:image/jpg;base64,{b64_image_2}'}
            }
        ]
    }
]

In [22]:
response = openai_client.chat.completions.create(
    model='gpt-5.4-mini',
    messages=messages
)

In [23]:
print(response.choices[0].message.content)

이 이미지는 넓고 고요한 바다를 배경으로, 하얀 절벽이나 바위 위에 사람들이 앉아 있는 풍경입니다.  

- **왼쪽 아래**에는 흰색의 거친 절벽이 보이고, 그 위에 몇 사람이 앉아 바다를 바라보고 있습니다.  
- **오른쪽 대부분**은 잔잔한 푸른 바다로 채워져 있어, 매우 평온한 분위기를 줍니다.  
- **하늘**은 연한 파란색에서 분홍빛, 주황빛으로 이어지는 **노을 또는 일출 무렵의 색감**처럼 보여서 아름답고 부드러운 느낌을 만듭니다.  

전체적으로 **한적하고 감성적인 해변 풍경**이며, 사람들은 바다를 감상하거나 사진을 찍고 있는 듯합니다.  
원하시면 이 이미지를 **더 자세히 묘사**하거나 **분위기 중심으로 해석**해드릴게요.


## 2개 이미지 비교를 요청하는 메시지 프롬트를 작성

In [24]:
# 이미지 파일들의 경로
cafe_1 = './images/cafe_1.jpg'
cafe_2 = './images/cafe_2.jpg'

In [25]:
# 이미지 바이너리 데이터를 ASCII 문자열로 인코딩.
b64_cafe_1 = base64encode_image(cafe_1)
b64_cafe_2 = base64encode_image(cafe_2)

In [26]:
messages = [
    {
        'role': 'user',
        'content': [
            {'type': 'text', 'text': '다음 2개 카페 이미지를 비교해줘'},
            {
                'type': 'image_url',
                'image_url': {'url': f'data:image/jpg;base64,{b64_cafe_1}'}
            },
            {
                'type': 'image_url',
                'image_url': {'url': f'data:image/jpg;base64,{b64_cafe_2}'}
            }
        ]
    }
]

In [27]:
response = openai_client.chat.completions.create(
    model='gpt-5.4-mini',
    messages=messages
)

In [28]:
print(response.choices[0].message.content)

두 카페 이미지를 비교해보면, 분위기와 인테리어 콘셉트가 꽤 다릅니다.

## 1) 전체 분위기
- **첫 번째 카페**:  
  식물과 자연광이 강하게 느껴지는 **정원/온실형 카페** 분위기입니다. 천장에 많은 식물이 매달려 있고, 밝고 싱그러운 인상이 큽니다.
- **두 번째 카페**:  
  나무, 철제 구조, 블랙 펜던트 조명 중심의 **빈티지·인더스트리얼 카페** 분위기입니다. 더 차분하고 도시적이며, 작업하기 좋은 느낌이 납니다.

## 2) 인테리어 스타일
- **첫 번째**
  - 대형 환기 덕트와 유리 지붕이 보이지만, 식물이 이를 부드럽게 덮어줌
  - 자연 소재 + 식물 중심
  - 밝고 개방적이며 장식성이 강함
- **두 번째**
  - 노출 천장, 검은 조명, 나무 테이블과 의자
  - 미니멀하면서도 러프한 공업적 느낌
  - 구조가 더 단정하고 정돈되어 보임

## 3) 채광과 색감
- **첫 번째**:  
  전체적으로 **밝은 톤, 초록색이 강조된 청량한 색감**입니다.
- **두 번째**:  
  **따뜻한 우드톤 + 회색 바닥 + 어두운 조명**으로 더 묵직하고 안정적인 색감입니다.

## 4) 좌석 및 공간감
- **첫 번째**
  - 테이블 간 간격이 비교적 넓어 보이고
  - 공간이 넓게 트여 있어 편안하고 여유로운 느낌
- **두 번째**
  - 좌석이 더 규칙적으로 배치되어 있고
  - 중앙 통로가 명확해서 실용적이며, 혼자 작업하거나 대화하기 좋을 것 같음

## 5) 이용 분위기
- **첫 번째**:  
  사진 찍기 좋고, 휴식/브런치/데이트에 어울리는 카페
- **두 번째**:  
  조용히 앉아 대화하거나 공부·작업하기 좋은 카페

## 한 줄 요약
- **첫 번째 카페 = 식물 가득한 밝은 온실형 감성 카페**
- **두 번째 카페 = 우드톤의 차분한 인더스트리얼 카페**

원하시면 제가 이어서  
**“어느 카페가 더 인스타 감성인지”**,  
**“데이트용으로 더 좋은 곳”**,  
**“공부하기 좋은 